# Historical Float Student Checkpoint Comparison

This notebook records the pre-prune audit that justified removing weak
float-domain student checkpoint variants from the default HydraNet catalog.

Current catalog contract:

- `anomaly_detection`, `burned_area`, `fire`, and `worldfloods` now keep only
  the strong `finetuning` `5000-shot` student checkpoint
- the `linear_probing` and lower-shot student variants shown below are
  historical audit data, not part of the current default workflow

Important scope:

- this is **not** a comparison of different model families from the public
  Phi2FM repo
- HydraNet currently ships one strong default student checkpoint per float
  expert; the comparisons below explain why the weaker variants were pruned

The notebook uses the corrected float export at
`outputs/routerset/fix30March_floatminmax_selected_anomalyfix/` and preserves
representative `train` and `validation` patches for the historical audit.


In [1]:
from __future__ import annotations

import contextlib
import io
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists() and (PROJECT_ROOT.parent / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from hydranet import load_student
from hydranet.moe_training import _reduce_expert_output_to_routing_score

# Historical audit notebook: current float-student defaults keep only FT-5000.

DATASET_ROOT = PROJECT_ROOT / 'outputs' / 'routerset' / 'fix30March_floatminmax_selected_anomalyfix'
MANIFEST_PATH = DATASET_ROOT / 'manifest_256.jsonl'
REPORT_PATH = DATASET_ROOT / 'student_checkpoint_comparison_summary.json'

EXPERTS = ['anomaly_detection', 'burned_area', 'fire', 'worldfloods']
VARIANTS = [
    {'label': 'FT-50', 'training': 'finetuning', 'n_shots': 50},
    {'label': 'LP-50', 'training': 'linear_probing', 'n_shots': 50},
    {'label': 'FT-5000', 'training': 'finetuning', 'n_shots': 5000},
    {'label': 'LP-5000', 'training': 'linear_probing', 'n_shots': 5000},
]
BASELINE_LABEL = 'FT-5000'
DEVICE = 'cpu'

torch.set_grad_enabled(False)
plt.rcParams['figure.figsize'] = (16, 8)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.max_open_warning'] = 0

print('project_root:', PROJECT_ROOT)
print('dataset_root:', DATASET_ROOT)
print('manifest_exists:', MANIFEST_PATH.exists())


project_root: /shared/home/rdelprete/PythonProjects/hydranet-phisat2
dataset_root: /shared/home/rdelprete/PythonProjects/hydranet-phisat2/outputs/routerset/fix30March_floatminmax_selected_anomalyfix


manifest_exists: True


In [2]:
FALSE_RGB_CHANNELS = (4, 2, 1)  # B08, B04, B03


def load_manifest_rows(path: Path) -> list[dict]:
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]


def resolve_path(value: str | Path) -> Path:
    path = Path(value)
    if path.is_absolute():
        return path
    return PROJECT_ROOT / path


def normalize_display(chw: np.ndarray, channels: tuple[int, int, int]) -> np.ndarray:
    rgb = np.stack([chw[index] for index in channels], axis=-1).astype(np.float32)
    lo = np.percentile(rgb, 2.0)
    hi = np.percentile(rgb, 98.0)
    if hi <= lo:
        return np.clip(rgb, 0.0, 1.0)
    rgb = (rgb - lo) / (hi - lo)
    return np.clip(rgb, 0.0, 1.0)


def infer_patch(model: torch.nn.Module, chw: np.ndarray) -> dict:
    x = torch.from_numpy(np.array(chw, copy=True)).unsqueeze(0).float().to(DEVICE)
    with torch.no_grad():
        logits = model(x)
        probs = torch.softmax(logits, dim=1)
    pred = probs.argmax(dim=1)[0].detach().cpu().numpy()
    conf = probs.max(dim=1).values[0].detach().cpu().numpy()
    score = _reduce_expert_output_to_routing_score(logits.detach().cpu())
    values, counts = np.unique(pred, return_counts=True)
    histogram = {int(v): int(c) for v, c in zip(values, counts)}
    return {
        'pred': pred,
        'conf': conf,
        'score': float(score),
        'histogram': histogram,
    }


def markdown_table(rows: list[dict], columns: list[tuple[str, str]]) -> str:
    header = '| ' + ' | '.join(label for _, label in columns) + ' |'
    divider = '| ' + ' | '.join(['---'] * len(columns)) + ' |'
    body = []
    for row in rows:
        body.append('| ' + ' | '.join(str(row[key]) for key, _ in columns) + ' |')
    return '\n'.join([header, divider, *body])


In [3]:
rows = load_manifest_rows(MANIFEST_PATH)
rows_by_expert_split: dict[tuple[str, str], list[dict]] = {}
for row in rows:
    expert = row['source_dataset']
    split = row.get('moe_split', row['source_split'])
    if expert not in EXPERTS:
        continue
    rows_by_expert_split.setdefault((expert, split), []).append(row)

selected_rows: list[dict] = []
for expert in EXPERTS:
    for split in ('train', 'validation'):
        group = sorted(
            rows_by_expert_split[(expert, split)],
            key=lambda row: (str(row['source_sample_id']), str(row['materialized_image_path'])),
        )
        selected_rows.append(group[0])

print(json.dumps([
    {
        'expert': row['source_dataset'],
        'split': row.get('moe_split', row['source_split']),
        'sample_id': row['source_sample_id'],
        'labels': row.get('label_names') or [],
        'materialized_image_path': row['materialized_image_path'],
    }
    for row in selected_rows
], indent=2))


[
  {
    "expert": "anomaly_detection",
    "split": "train",
    "sample_id": "0000000",
    "labels": [
      "water"
    ],
    "materialized_image_path": "outputs/routerset/fix30March_floatminmax_selected_anomalyfix/images/anomaly_detection/train/0000000_0_0_256_256.npy"
  },
  {
    "expert": "anomaly_detection",
    "split": "validation",
    "sample_id": "0000001",
    "labels": [
      "cloud",
      "land",
      "marine_algae",
      "turbid_water"
    ],
    "materialized_image_path": "outputs/routerset/fix30March_floatminmax_selected_anomalyfix/images/anomaly_detection/validation/0000001_0_0_256_256.npy"
  },
  {
    "expert": "burned_area",
    "split": "train",
    "sample_id": "0000002",
    "labels": [],
    "materialized_image_path": "outputs/routerset/fix30March_floatminmax_selected_anomalyfix/images/burned_area/train/0000002_0_0_256_256.npy"
  },
  {
    "expert": "burned_area",
    "split": "validation",
    "sample_id": "0000075",
    "labels": [],
    "materializ

In [4]:
models = {}
model_logs = {}
for expert in EXPERTS:
    for variant in VARIANTS:
        key = (expert, variant['training'], variant['n_shots'])
        buffer = io.StringIO()
        with contextlib.redirect_stdout(buffer):
            model = load_student(
                task=expert,
                training=variant['training'],
                n_shots=variant['n_shots'],
                auto_load_weights=True,
            )
        model.eval()
        model.to(DEVICE)
        models[key] = model
        model_logs[key] = [line for line in buffer.getvalue().splitlines() if line.strip()]

for expert in EXPERTS:
    print(f'## {expert}')
    for variant in VARIANTS:
        key = (expert, variant['training'], variant['n_shots'])
        print(f"{variant['label']}: {model_logs[key][-1] if model_logs[key] else 'loaded'}")
    print()


artifacts.json: 0.00B [00:00, ?B/s]

artifacts.json: 0.00B [00:00, ?B/s]

finetuning/hydranet/anomaly_detection_ns(…):   0%|          | 0.00/1.44M [00:00<?, ?B/s]

artifacts.json: 0.00B [00:00, ?B/s]

artifacts.json: 0.00B [00:00, ?B/s]

...

artifacts.json: 0.00B [00:00, ?B/s]

finetuning/hydranet/worldfloods_nshot50_(…):   0%|          | 0.00/1.44M [00:00<?, ?B/s]

linear_probing/hydranet/worldfloods_nsho(…):   0%|          | 0.00/1.44M [00:00<?, ?B/s]

linear_probing/hydranet/worldfloods_nsho(…):   0%|          | 0.00/1.44M [00:00<?, ?B/s]

## anomaly_detection
FT-50:   Adjusted n_classes to checkpoint head: 9
LP-50:   Adjusted n_classes to checkpoint head: 9
FT-5000:   Adjusted n_classes to checkpoint head: 9
LP-5000:   Adjusted n_classes to checkpoint head: 9

## burned_area
FT-50:   Adjusted n_classes to checkpoint head: 4
LP-50:   Adjusted n_classes to checkpoint head: 4
FT-5000:   Adjusted n_classes to checkpoint head: 4
LP-5000:   Adjusted n_classes to checkpoint head: 4

## fire
FT-50:   Note: 4 keys in checkpoint not used (e.g., task-specific heads)
LP-50:   Note: 4 keys in checkpoint not used (e.g., task-specific heads)
FT-5000:   Note: 4 keys in checkpoint not used (e.g., task-specific heads)
LP-5000:   Note: 4 keys in checkpoint not used (e.g., task-specific heads)

## worldfloods
FT-50: Loading weights from: /shared/home/rdelprete/.cache/huggingface/hub/datasets--sirbastiano94--hydranet/snapshots/a2ae755b4f5ddd2e25a9292c677b7c1f6616e065/finetuning/hydranet/worldfloods_nshot50_frozen/worldfloods/20251212_UNet_M

In [5]:
report = {'dataset_root': str(DATASET_ROOT), 'baseline': BASELINE_LABEL, 'per_patch': []}

for row in selected_rows:
    expert = row['source_dataset']
    split = row.get('moe_split', row['source_split'])
    image = np.load(resolve_path(row['materialized_image_path'])).astype(np.float32, copy=False)

    outputs = {}
    for variant in VARIANTS:
        key = (expert, variant['training'], variant['n_shots'])
        outputs[variant['label']] = infer_patch(models[key], image)

    baseline = outputs[BASELINE_LABEL]
    table_rows = []
    for variant in VARIANTS:
        result = outputs[variant['label']]
        disagree = float(np.mean(result['pred'] != baseline['pred']))
        conf_delta = float(np.mean(result['conf'] - baseline['conf']))
        table_rows.append(
            {
                'variant': variant['label'],
                'score': f"{result['score']:.4f}",
                'mean_conf': f"{float(np.mean(result['conf'])):.4f}",
                'disagree_vs_ft5000': f"{disagree:.4f}",
                'conf_delta_vs_ft5000': f"{conf_delta:.4f}",
                'histogram': result['histogram'],
            }
        )

    report['per_patch'].append(
        {
            'expert': expert,
            'split': split,
            'sample_id': row['source_sample_id'],
            'metrics': table_rows,
        }
    )

    display(Markdown(f"## {expert} | {split} | `{row['source_sample_id']}`"))

    fig, axes = plt.subplots(1, 5, figsize=(20, 4.5))
    fig.suptitle(f"{expert} | {split} | sample={row['source_sample_id']}", fontsize=14)

    axes[0].imshow(normalize_display(image, FALSE_RGB_CHANNELS))
    axes[0].set_title('False RGB')

    for index, variant in enumerate(VARIANTS, start=1):
        result = outputs[variant['label']]
        axes[index].imshow(result['pred'], cmap='tab20')
        axes[index].set_title(f"{variant['label']}\nscore={result['score']:.4f}")

    for ax in axes:
        ax.set_xticks([])
        ax.set_yticks([])

    plt.tight_layout(rect=(0, 0, 1, 0.94))
    plt.show()

    display(
        Markdown(
            markdown_table(
                table_rows,
                [
                    ('variant', 'variant'),
                    ('score', 'routing score'),
                    ('mean_conf', 'mean confidence'),
                    ('disagree_vs_ft5000', 'disagree vs FT-5000'),
                    ('conf_delta_vs_ft5000', 'conf delta vs FT-5000'),
                    ('histogram', 'pred histogram'),
                ],
            )
        )
    )

REPORT_PATH.write_text(json.dumps(report, indent=2), encoding='utf-8')
print('summary_report:', REPORT_PATH)


## anomaly_detection | train | `0000000`

| variant | routing score | mean confidence | disagree vs FT-5000 | conf delta vs FT-5000 | pred histogram |
| --- | --- | --- | --- | --- | --- |
| FT-50 | 0.1153 | 0.2076 | 0.0000 | -0.0179 | {1: 65536} |
| LP-50 | 0.1190 | 0.2069 | 0.0000 | -0.0187 | {1: 65536} |
| FT-5000 | 0.1240 | 0.2255 | 0.0000 | 0.0000 | {1: 65536} |
| LP-5000 | 0.1246 | 0.2276 | 0.8417 | 0.0021 | {1: 10377, 7: 55159} |

## anomaly_detection | validation | `0000001`

| variant | routing score | mean confidence | disagree vs FT-5000 | conf delta vs FT-5000 | pred histogram |
| --- | --- | --- | --- | --- | --- |
| FT-50 | 0.1152 | 0.2004 | 0.0000 | -0.0191 | {1: 65536} |
| LP-50 | 0.1187 | 0.2014 | 0.0000 | -0.0181 | {1: 65536} |
| FT-5000 | 0.1238 | 0.2195 | 0.0000 | 0.0000 | {1: 65536} |
| LP-5000 | 0.1245 | 0.2186 | 0.1836 | -0.0009 | {1: 53503, 7: 12033} |

## burned_area | train | `0000002`

...

## worldfloods | train | `0000071`

| variant | routing score | mean confidence | disagree vs FT-5000 | conf delta vs FT-5000 | pred histogram |
| --- | --- | --- | --- | --- | --- |
| FT-50 | 0.3505 | 0.4090 | 0.0000 | -0.0485 | {1: 65536} |
| LP-50 | 0.3562 | 0.4235 | 0.0000 | -0.0340 | {1: 65536} |
| FT-5000 | 0.3655 | 0.4575 | 0.0000 | 0.0000 | {1: 65536} |
| LP-5000 | 0.3684 | 0.4323 | 0.0000 | -0.0252 | {1: 65536} |

## worldfloods | validation | `0000803`

| variant | routing score | mean confidence | disagree vs FT-5000 | conf delta vs FT-5000 | pred histogram |
| --- | --- | --- | --- | --- | --- |
| FT-50 | 0.3505 | 0.4093 | 0.0000 | -0.0488 | {1: 65536} |
| LP-50 | 0.3560 | 0.4239 | 0.0000 | -0.0342 | {1: 65536} |
| FT-5000 | 0.3656 | 0.4581 | 0.0000 | 0.0000 | {1: 65536} |
| LP-5000 | 0.3683 | 0.4338 | 0.0000 | -0.0243 | {1: 65536} |

summary_report: /shared/home/rdelprete/PythonProjects/hydranet-phisat2/outputs/routerset/fix30March_floatminmax_selected_anomalyfix/student_checkpoint_comparison_summary.json
